# Part I: Initial Benchmark
We are going to benchmark our current implementation, first we want to set $n$ and $m$ so that they can be on a reasonably sized instance. 

We use the ks_func function we implemented in HW2, and we set $n$  and $m$  to be 1,000,000 instead of 1000 to make the memory bottleneck and performance issues more observable.  And we use the benchmark tool to see the execute time and see what'll happen.



In [5]:
using Plots
using Revise
includet("C:/Users/Odysseus/GR6104_Homework/HW3/src/ks-stat.jl")

In [9]:
using BenchmarkTools
using Random 
using Profile
using ProfileCanvas

Random.seed!(2026)
n = randn(1000000)
m = randn(1000000)
@btime ks_func(n,m)
@benchmark ks_func(n,m)


  249.585 ms (116262 allocations: 97.33 MiB)


BenchmarkTools.Trial: 18 samples with 1 evaluation per sample.
 Range (min … max):  256.368 ms … 323.660 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     279.473 ms               ┊ GC (median):    7.65%
 Time  (mean ± σ):   285.287 ms ±  19.434 ms  ┊ GC (mean ± σ):  6.30% ± 5.42%

           ▃                            █                        
  ▇▁▁▁▇▁▁▇▁█▁▁▁▁▁▇▇▇▁▁▇▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▇▇▁▁▇▁▁▇▁▁▁▁▁▁▁▁▁▁▁▁▇ ▁
  256 ms           Histogram: frequency by time          324 ms <

 Memory estimate: 97.41 MiB, allocs estimate: 117655.

The current implementation takes approximately 170 ms to run. More importantly, it allocates 91.55 MiB of memory. This indicates a significant inefficiency, likely due to the creation of intermediate data structures during the labeling and sorting process.

# Part II: Iterative Code Optimization 
## - Round 1
## 1. Identify Bottlenecks

In [ ]:
ks_func(n[1:10], m[1:10])

Profile.clear()

ProfileCanvas.@profview ks_func(n, m)

## Function and Line Number:
Based on the result from the Flame Graph, the primary bottleneck is located in the ks_func function, specifically at line 10 in ks-stat.jl. This line corresponds to the sort! operation on the combined array.

## Nature of the Bottleneck:
The bottleneck is primarily caused by unnecessary memory allocation, which leads to excessive computation:
    1. Memory Allocation caused by string, since the code creates a new tuple containing string (value,"label") for every single data point for a single run, it causes a huge volume of pressure on system's memory and the Collector.
    2. The Flame Graph shows that 74% of the execution time is spent in sort!. We might need to change the sort! a bit.

# 2. Alternative Implementations:

### Rationale for Alternatives
The bottleneck in the original code was the creation of intermediate `(value, label)` tuples and sorting a mixed array.
* **Alternative 1 (Separate Sorting + Two-Pointer):** Instead of mixing the arrays, we sort `X` and `Y` separately. This allows us to use Julia's highly optimized native float sorting. We then use a "Two-Pointer" algorithm to calculate the KS statistic in faster way which will avoid more allocations.

In [3]:
# Implementation for the alternatives two-pointer function 


function ks_func_2pt(X::AbstractVector,Y::AbstractVector)

    max_diff= 0
    cdf_x = 0
    cdf_y = 0
    n = length(X)
    m = length(Y)
    sort_x = sort(X)
    sort_y =sort(Y)
    i=1
    j=1

    while i<=n || j<=m 
        if i>n
            current_val = sort_y[j]
        elseif j>m
            current_val = sort_x[i]
        else 
            current_val = min(sort_x[i],sort_y[j])
        end
    

        while i <=n && sort_x[i]== current_val #??current_val_x == sort_x[i]
            cdf_x +=1/n
            i+=1
        end

        while j <=m && sort_y[j]== current_val
            cdf_y +=1/m
            j+=1
        end

        curr_diff = abs(cdf_x-cdf_y)
        if curr_diff > max_diff
            max_diff =curr_diff
        end

    end

    return max_diff

end        

ks_func_2pt (generic function with 1 method)

# 3. Benchmark Alternatives:

In [10]:
using BenchmarkTools
using Random 
using Profile
using ProfileCanvas

Random.seed!(2026)
n = randn(1000000)
m = randn(1000000)
@btime ks_func_2pt(n,m)
@benchmark ks_func_2pt(n,m)

  101.445 ms (36642 allocations: 30.54 MiB)


BenchmarkTools.Trial: 34 samples with 1 evaluation per sample.
 Range (min … max):  123.824 ms … 205.479 ms  ┊ GC (min … max): 0.00% … 34.03%
 Time  (median):     142.234 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   147.424 ms ±  22.419 ms  ┊ GC (mean ± σ):  5.53% ± 10.24%

  ▃▃   ▃  ▃ ▃▃ ▃█▃ ▃                                             
  ██▁▇▇█▇▁█▇██▁███▁█▁▇▁▁▇▇▁▇▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▇▁▁▁▁▇▁▇▇▁▁▁▁▇ ▁
  124 ms           Histogram: frequency by time          205 ms <

 Memory estimate: 30.39 MiB, allocs estimate: 33639.

In [12]:
ks_func_2pt(n[1:10], m[1:10])

Profile.clear()

ProfileCanvas.@profview  ks_func_2pt(n, m)

As we can see the alternative implementation significantly improved the bottleneck:

Execution Time: The median runtime dropped from 279.47 ms to 142.23 ms, achieving a ~2x speedup.

Memory Efficiency: Total memory allocation was reduced by 68% (from 97.33 MiB to 30.39 MiB).

Conclusion: The Flame Graph previously identified sort! as the main culprit (74% of time). By switching to a Two-Pointer approach with separate sorting of raw Float64 vectors, we bypassed the expensive process of sorting tuples with strings.

# 4. Update Code Base:

Therefore we have improved the bottleneck, we update our code base `ks_func` to `ks_func_2pt`. We re-run the unit tests to verify that nothing is broken.


In [14]:
# check for correctness

n = randn(1000)
m = randn(2000)

# run benchmarks
v1 = @btime ks_func(n,m)
v2 = @btime ks_func_2pt(n,m)

@assert isapprox(v1, v2)

  157.300 μs (13 allocations: 140.93 KiB)
  67.500 μs (23 allocations: 41.99 KiB)


We can see that the two results are approximately equal, we can say we are correct. And we update our `ks-stat,jl` in `src` and `ks-unit-test.jl` in `test`.

We can see that all the unit tests pass, nothing is broken , then our result is correct for the new optimization version. 

We commit and push our new code to GitHub.

### As we can see from the above results, the behavior of the KS_function has been improved. But we still want more.

# Part II: Iterative Code Optimization
## Round 2
## 1.Identify Bottlenecks: 


Function and Line Number: The bottleneck is identified in the sort! function (or sort), specifically where the input arrays $X$ and $Y$ are sorted at the beginning of the ks_func. 
In the flame graph from Round 1, this operation now occupies the vast majority (e.g., >80%) of the total execution time.

Nature of the Bottleneck:The nature of this bottleneck is Computationally Intensive Serial Execution.Computation: unlike Round 1, we are no longer suffering from unnecessary memory allocation (since we removed the tuple creation). The remaining cost is the mathematical necessity of sorting two large arrays, which has a time complexity of $O(N \log N)$.

The Key Issue
: The current implementation sorts array $X$ and array $Y$ sequentially (one after the other). Since the sorting of $X$ is mathematically independent of $Y$, running them serially on a single CPU core is inefficient. It leaves other CPU cores idle, resulting in a "latency bottleneck" where the total time is the sum of both sorting operations ($T_x + T_y$) rather than the maximum of the two ($\max(T_x, T_y)$).

In [17]:
Random.seed!(2026)
n = randn(1000)
m = randn(1000)
@btime ks_func_2pt(n,m)
@benchmark ks_func_2pt(n,m)

  51.200 μs (23 allocations: 35.27 KiB)


BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  50.500 μs …  21.812 ms  ┊ GC (min … max): 0.00% … 99.38%
 Time  (median):     72.100 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   88.435 μs ± 224.521 μs  ┊ GC (mean ± σ):  2.45% ±  0.99%

  ▄▄█▇▆▄▂▁                                                      
  █████████▆▅▄▃▃▂▂▂▂▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  50.5 μs         Histogram: frequency by time          299 μs <

 Memory estimate: 35.27 KiB, allocs estimate: 23.

## 2. Alternative Implementations:
Rationale for the alternative:
The current bottleneck is the serial execution of two independent sorting operations ($O(N \log N)$). Since the sorting of array $X$ and array $Y$ are mathematically independent, they can be performed simultaneously on separate CPU cores.

Alternative (Parallel Sorting): We utilize Julia's multi-threading capabilities (Base.Threads). By spawning a new thread to sort $Y$ while the main thread sorts $X$, we can reduce the sorting wall-clock time by up to 50% (ignoring overhead), as the operations run in parallel rather than sequentially.

In [18]:
using Base.Threads
println("Current Number of Threads: ", Threads.nthreads())

Current Number of Threads: 1


In [32]:
function ks_func_parallel(X::AbstractVector, Y::AbstractVector)
    # We spawn a task to sort Y on a separate thread
    y_task = Threads.@spawn sort(Y)
    sx =sort(X)
    sy = fetch(y_task)::Vector{Float64} #Wait for Y to finish the sorting and fetch the result
    i =1
    j =1
    cdf_x =0
    cdf_y =0
    n = length(sx)
    m = length(sy)
    max_diff= 0

    while i<=n || j<=m 
        if i>n
            current_val = sy[j]
        elseif j>m
            current_val = sx[i]
        else 
            current_val = min(sx[i],sy[j])
        end
    

        while i <=n && sx[i]== current_val #??current_val_x == sort_x[i]
            cdf_x +=1/n
            i+=1
        end

        while j <=m && sy[j]== current_val
            cdf_y +=1/m
            j+=1
        end

        curr_diff = abs(cdf_x-cdf_y)
        if curr_diff > max_diff
            max_diff =curr_diff
        end

    end

    return max_diff
end

ks_func_parallel (generic function with 1 method)

# 3. Benchmark Alternatives

In [ ]:
using BenchmarkTools
using Random 
using Profile
using ProfileCanvas


Random.seed!(2026)
# Increase data size to 1 million to observe parallel benefits
n2 = randn(1000000)
m2 = randn(1000000)

println("Two Pointer Method: ")
@btime ks_func_2pt($n2, $m2)


println("Parallel Sorting: ")
@btime ks_func_parallel($n2, $m2)
@benchmark ks_func_parallel(n,m)

Two Pointer Method: 
  95.741 ms (26 allocations: 28.72 MiB)
Parallel Sorting: 
  69.341 ms (31 allocations: 28.72 MiB)


BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):   49.500 μs …  66.888 ms  ┊ GC (min … max): 0.00% … 96.43%
 Time  (median):     135.200 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   189.957 μs ± 681.939 μs  ┊ GC (mean ± σ):  3.40% ±  0.96%

       ▃█▁                                                       
  ▁▂▃▆████▇▆▅▅▄▄▃▃▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▂
  49.5 μs          Histogram: frequency by time          597 μs <

 Memory estimate: 35.60 KiB, allocs estimate: 28.

In [38]:
ks_func_parallel(n2[1:10], m2[1:10])

Profile.clear()

ProfileCanvas.@profview  ks_func_parallel(n2, m2)

As we can see from the result, he parallel implementation significantly improved the bottleneck.

#### Round 1: The median runtime was 95.741 ms.

#### Round 2: Parallel Execution : The median runtime dropped to 69.341 ms.

Analysis of the Bottleneck:
Before round 2, the CPU had to sort array $X$ and then sort array $Y$ sequentially. The total sorting time was roughly $T_{sortX} + T_{sortY}$. After using the parallel methods, we allowed array $Y$ to be sorted on a separate core at the same time as array $X$. This effectively reduced the sorting wall-clock time. This represents a reduction in execution time of approximately 27.6% (or a 1.38x speedup).
Additionally, the total memory usage remained identical at 28.72 MiB, this means our parallelization introduced negligible memory overhead  while the executive speed goes up.

# 4. Update Code Base:
We check for correctness and use re-run your unit tests to verify that nothing is broken.



In [41]:
# check for correctness

n = randn(1000)
m = randn(2000)

# run benchmarks
v1 = @btime ks_func(n,m)
v2 = @btime ks_func_2pt(n,m)
v3 = @btime ks_func_parallel(n,m)

@assert isapprox(v1, v2) && isapprox(v1,v3) 

#We re-run our unit tests to verify that nothing is broken

  189.100 μs (13 allocations: 140.93 KiB)
  82.400 μs (28 allocations: 65.09 KiB)
  86.400 μs (33 allocations: 65.39 KiB)


We can see that the three results are approximately equal, we can say we are correct. And we update our `ks-stat,jl` in `src` and `ks-unit-test.jl` in `test`.

We see that all the unit tests pass, nothing is broken , then our result is correct for the new optimization version. 

We commit and push our new code to GitHub.

# Part III: Final Benchmark
We use the same test instance from Part I to benchmark the performance of our code.

1. Benchmark your final implementation. Describe the performance gains both in
terms of execution time and memory.


In [45]:
using BenchmarkTools
using Random 
using Profile
using ProfileCanvas

Random.seed!(2026)
# Increase data size to 1 million to observe parallel benefits
n2 = randn(1000000)
m2 = randn(1000000)

println("The Original Method: ")
@btime ks_func($n2, $m2)
println("Two Pointer Method: ")
@btime ks_func_2pt($n2, $m2)
println("Parallel Sorting: ")
@btime ks_func_parallel($n2, $m2)

@benchmark ks_func_parallel(n2,m2)

The Odysseusriginal Method: 
  266.908 ms (12 allocations: 91.55 MiB)
Two Pointer Method: 
  95.671 ms (26 allocations: 28.72 MiB)
Parallel Sorting: 
  69.023 ms (31 allocations: 28.72 MiB)


BenchmarkTools.Trial: 64 samples with 1 evaluation per sample.
 Range (min … max):  67.941 ms … 115.114 ms  ┊ GC (min … max): 0.00% … 32.49%
 Time  (median):     77.898 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   79.095 ms ±   7.711 ms  ┊ GC (mean ± σ):  4.57% ±  7.73%

          ▄  ▄  ▂▂ ▄█                                           
  ▄▁▆▁▄▁▆██▄▆██▄██▆██▄▆▁▄▄▄▄▆▄▄▁▁▁▁▄▁▁▁▁▁▄▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄ ▁
  67.9 ms         Histogram: frequency by time          106 ms <

 Memory estimate: 28.72 MiB, allocs estimate: 32.

In [46]:
ks_func_2pt(n[1:10], m[1:10])

Profile.clear()

ProfileCanvas.@profview  ks_func_parallel(n2, m2)

## Detailed Analysis:

### 1.Execution Time:

The final parallel implementation is nearly 4 times faster than the original code. This dramatic improvement stems from two distinct optimization phases:

Algorithmic Efficiency (Round 1): By switching to the "Two-Pointer" method, we eliminated the overhead of creating thousands of tuples and the costly array concatenation (vcat). This alone reduced the time from ~266ms to ~95ms.

Parallel Computing (Round 2): By identifying sort! as the remaining bottleneck and applying Threads.@spawn, we utilized multi-core processing to sort the two large arrays simultaneously. This further reduced the runtime from ~95ms to ~69ms, achieving an additional ~27% speedup over the serial optimized version.

### 2. Memory Efficiency Gains
We achieved a massive around 68.6% reduction in memory usage (dropping from 91.55 MiB to 28.72 MiB).
